# knime2py — KNIME → Python workbook

**Workflow:** `INZ_visa_decisions_model__g01`  
**Source:** `/work/INZ_visa_decisions_model/workflow.knime`

This file was generated by the open-source **knime2py** project (KNIME → Python codegen).  
**GitHub:** https://github.com/vitalii-kaplan/knime2py

**Notes**
- The code below is a linear translation of KNIME nodes into Python sections.
- A lightweight `context` dict is available for debugging/inspection of intermediate tables.


**Export coverage**

- All nodes successfully exported.


In [55]:
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy import stats as _scipy_stats
import csv
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import re as _re
import tempfile


In [56]:
# Shared context to pass dataframes/tables between nodes (for debugging)
context = {}


## CSV Reader \# `1591`
Node state: `EXECUTED`  
comments: Combined  
Output: [Port 1] 1591:1 to 1559 Normalizer #1559  


In [57]:
################################################################################################################################################################
## CSV Reader # `1591`
# Node state: `EXECUTED`
# Output: [Port 1] 1591:1 to 1559 Normalizer #1559
# comments: Combined
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.io.filehandling.csv.reader.CSVTableReaderNodeFactory
csv_path = Path(r"data/prepared/inz_2022-2025_wb_who_m49_combined.csv")
# Note: escapechar equals quotechar; omitting escapechar and relying on double-quoted escapes.
df = pd.read_csv(csv_path, sep=',', quotechar='"', header=0, encoding='UTF-8', na_values=['', ' '], keep_default_na=True, skipinitialspace=True)
_pd_dtypes = {'Country': 'string', 'Approval rate': 'Float64', 'Applications_log10': 'Float64', 'WHO_Region': 'string', 'WHO_Under5_Mortality_per_1000_live_births': 'Float64', 'WHO_Maternal_Mortality_per_100000_live_births': 'Float64', 'WHO_Physician_Density_per_10000_population': 'Float64', 'WHO_DTP3_Immunization_Coverage_pct': 'Int64', 'WB_FY2026_Region': 'string', 'WB_FY2026_Income_Group': 'string', 'WB_FY2026_Lending_Category': 'string', 'GDP_per_capita_current_USD': 'Float64', 'Life_expectancy_at_birth_years': 'Float64', 'Internet_users_pct_population': 'Float64', 'Urban_population_pct': 'Float64', 'GNI_per_capita_Atlas_current_USD': 'Int64', 'UN M49': 'string'}
for _col, _dt in _pd_dtypes.items():
    if _col not in df.columns:
        continue
    try:
        if _dt in ('Int64', 'Float64'):
            df[_col] = pd.to_numeric(df[_col], errors='coerce').astype(_dt)
        else:
            df[_col] = df[_col].astype(_dt)
    except Exception:  # leave column as-is on failure
        pass
context['1591:1'] = df


## Normalizer \# `1559`
Node state: `EXECUTED`  
Input: [Port 1] 1591:1 from CSV Reader #1591  
Output: [Port 1] 1559:1 to 1574 Regression Predictor #1574; [Port 1] 1559:1 to 1576 Linear Regression Learner #1576  


In [58]:
################################################################################################################################################################
## Normalizer # `1559`
# Node state: `EXECUTED`
# Input: [Port 1] 1591:1 from CSV Reader #1591
# Output: [Port 1] 1559:1 to 1574 Regression Predictor #1574; [Port 1] 1559:1 to 1576 Linear Regression Learner #1576
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.preproc.normalize3.Normalizer3NodeFactory
df = context['1591:1']  # input table
out_df = df.copy()
bundle = {
    'mode': 'MINMAX',
    'new_min': 0.0,
    'new_max': 1.0,
    'excludes': ['Approval rate', 'Applications_log10'],
    'columns': [],
    'stats': {},
}
all_cols = out_df.columns.tolist()
cand_cols = all_cols
exclude_cols = ['Approval rate', 'Applications_log10']
cand_cols = [c for c in cand_cols if c not in set(exclude_cols)]
norm_cols = out_df[cand_cols].select_dtypes(include=['number', 'bool', 'boolean', 'Int64', 'Float64']).columns.tolist()
bundle['columns'] = list(norm_cols)
if not norm_cols:
    # No numeric columns to normalize; passthrough
    bundle['stats'] = {}
    pass
else:
    # Coerce selected columns to numeric before normalization
    out_df[norm_cols] = out_df[norm_cols].apply(pd.to_numeric, errors='coerce')
    _new_min, _new_max = 0.0, 1.0
    _span = (_new_max - _new_min)
    _col_min = out_df[norm_cols].min(axis=0, skipna=True)
    _col_max = out_df[norm_cols].max(axis=0, skipna=True)
    stats = {}
    for col in norm_cols:
        mn = _col_min.get(col)
        mx = _col_max.get(col)
        stats[col] = {
            'min': None if pd.isna(mn) else float(mn),
            'max': None if pd.isna(mx) else float(mx),
        }
    bundle['stats'] = stats
    def _minmax_col(s):
        mn = _col_min.get(s.name)
        mx = _col_max.get(s.name)
        rng = (mx - mn) if (mn is not None and mx is not None) else None
        if rng is None or pd.isna(rng) or rng == 0:
            # constant/empty column → map to new_min
            return pd.Series([_new_min] * len(s), index=s.index)
        return (_new_min + (s - mn) / rng * _span).astype(float)
    out_df[norm_cols] = out_df[norm_cols].apply(_minmax_col)
context['1559:1'] = out_df


## Linear Regression Learner \# `1576`
Node state: `EXECUTED`  
Input: [Port 1] 1559:1 from Normalizer #1559  
Output: [Port 1] 1576:1 to 1574 Regression Predictor #1574; [Port 2] 1576:2 to 1587 CSV Writer #1587  


In [ ]:
################################################################################################################################################################
## Linear Regression Learner # `1576`
# Node state: `EXECUTED`
# Input: [Port 1] 1559:1 from Normalizer #1559
# Output: [Port 1] 1576:1 to 1574 Regression Predictor #1574; [Port 2] 1576:2 to 1587 CSV Writer #1587
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.mine.regression.linear2.learner.LinReg2LearnerNodeFactory2
df = context['1559:1']  # input table
out_df = df.copy()
_target = 'Approval rate'
if _target not in df.columns:
    raise KeyError(f'Linear Regression Learner: target column not found: {_target!r}')
_include_cols = ['WHO_Under5_Mortality_per_1000_live_births', 'WHO_Maternal_Mortality_per_100000_live_births', 'WHO_Physician_Density_per_10000_population', 'WHO_DTP3_Immunization_Coverage_pct', 'WB_FY2026_Income_Group', 'WB_FY2026_Lending_Category', 'GDP_per_capita_current_USD', 'Life_expectancy_at_birth_years', 'Internet_users_pct_population', 'Urban_population_pct', 'GNI_per_capita_Atlas_current_USD',, 'UN M49']
source_cols = [c for c in _include_cols if c in df.columns and c != _target]
_exclude_cols = ['Country', 'Applications_log10', 'WHO_Region', 'WB_FY2026_Region']
source_cols = [c for c in source_cols if c not in set(_exclude_cols)]
if not source_cols:
    raise ValueError('Linear Regression Learner: no feature columns selected')
_include_constant = True
_missing_mode = 'fail'

feature_info = []
x_parts = []
for col in source_cols:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
        vals = pd.to_numeric(s, errors='coerce').astype(float)
        x_parts.append(vals.to_frame(col))
        feature_info.append({'kind': 'numeric', 'column': col, 'features': [col]})
    else:
        cat = s.astype('object').where(s.notna(), 'Missing').astype(str)
        dummies = pd.get_dummies(cat, prefix=col, prefix_sep='=', dtype=float)
        drop_level = None
        if len(dummies.columns) > 0:
            drop_col = dummies.columns[0]
            drop_level = str(drop_col).split('=', 1)[1] if '=' in str(drop_col) else str(drop_col)
            dummies = dummies.iloc[:, 1:]
        if len(dummies.columns) > 0:
            x_parts.append(dummies)
        feature_info.append({
            'kind': 'categorical',
            'column': col,
            'levels': sorted(cat.dropna().unique().tolist()),
            'drop_level': drop_level,
            'features': list(dummies.columns),
        })

X_df = pd.concat(x_parts, axis=1) if x_parts else pd.DataFrame(index=df.index)
X_df = X_df.astype(float)
y = pd.to_numeric(df[_target], errors='coerce').astype(float)
valid_mask = y.notna() & ~X_df.isna().any(axis=1)
if _missing_mode == 'fail' and not bool(valid_mask.all()):
    raise ValueError('Linear Regression Learner: missing values found in target or features')
X_fit = X_df.loc[valid_mask].copy()
y_fit = y.loc[valid_mask].copy()
if X_fit.empty:
    raise ValueError('Linear Regression Learner: no training rows available')

design_cols = list(X_fit.columns)
X_mat = X_fit.to_numpy(dtype=float)
if _include_constant:
    X_mat = np.column_stack([X_mat, np.ones(len(X_fit), dtype=float)])
    design_cols_with_intercept = design_cols + ['Intercept']
else:
    design_cols_with_intercept = design_cols
y_vec = y_fit.to_numpy(dtype=float)
coef = np.linalg.lstsq(X_mat, y_vec, rcond=None)[0]
pred_fit = X_mat @ coef
resid = y_vec - pred_fit
df_resid = int(max(len(y_vec) - X_mat.shape[1], 0))
if df_resid > 0:
    sigma2 = float((resid @ resid) / df_resid)
    cov = sigma2 * np.linalg.pinv(X_mat.T @ X_mat)
    std_err = np.sqrt(np.diag(cov))
    t_vals = np.divide(coef, std_err, out=np.full_like(coef, np.nan, dtype=float), where=std_err != 0)
    p_vals = 2 * _scipy_stats.t.sf(np.abs(t_vals), df_resid)
else:
    std_err = np.full_like(coef, np.nan, dtype=float)
    t_vals = np.full_like(coef, np.nan, dtype=float)
    p_vals = np.full_like(coef, np.nan, dtype=float)
coef_df = pd.DataFrame({
    'Variable': design_cols_with_intercept,
    'Coeff.': coef,
    'Std. Err.': std_err,
    't-value': t_vals,
    'P>|t|': p_vals,
})
summary_df = pd.DataFrame([{'n_rows': len(y_vec), 'n_features': len(design_cols), 'df_resid': df_resid, 'r_squared': float(1 - (resid @ resid) / np.sum((y_vec - y_vec.mean()) ** 2)) if len(y_vec) > 1 and np.sum((y_vec - y_vec.mean()) ** 2) != 0 else np.nan}])
model_bundle = {
    'kind': 'linear_regression',
    'target': _target,
    'source_cols': list(source_cols),
    'features': list(design_cols),
    'feature_cols': list(design_cols),
    'feature_info': feature_info,
    'include_constant': _include_constant,
    'coef': coef.tolist(),
    'coefficients': coef.tolist(),
    'intercept': float(coef[-1]) if _include_constant and len(coef) else 0.0,
}
context['1576:1'] = model_bundle
context['1576:2'] = coef_df


## Regression Predictor \# `1574`
Node state: `EXECUTED`  
Input: [Port 2] 1559:1 from Normalizer #1559; [Port 1] 1576:1 from Linear Regression Learner #1576  
Output: [Port 1] 1574:1 to 1571 Column Filter #1571; [Port 1] 1574:1 to 1577 Numeric Scorer #1577  


In [60]:
################################################################################################################################################################
## Regression Predictor # `1574`
# Node state: `EXECUTED`
# Input: [Port 2] 1559:1 from Normalizer #1559; [Port 1] 1576:1 from Linear Regression Learner #1576
# Output: [Port 1] 1574:1 to 1571 Column Filter #1571; [Port 1] 1574:1 to 1577 Numeric Scorer #1577
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.mine.regression.predict3.RegressionPredictorNodeFactory2
model_key = '1576:1'
data_key = '1559:1'
model_obj = context[model_key]
df = context[data_key]
out_df = df.copy()
bundle = model_obj if isinstance(model_obj, dict) else {'estimator': model_obj}
target = bundle.get('target') or bundle.get('target_name') or bundle.get('y_col')
pred_col = f'Prediction ({target or "target"})'

if bundle.get('kind') == 'linear_regression' or 'coef' in bundle or 'coefficients' in bundle:
    feature_info = list(bundle.get('feature_info') or [])
    x_parts = []
    for info in feature_info:
        col = info.get('column')
        if col not in out_df.columns:
            raise KeyError(f'Regression Predictor: missing feature column: {col!r}')
        if info.get('kind') == 'numeric':
            feat_name = (info.get('features') or [col])[0]
            vals = pd.to_numeric(out_df[col], errors='coerce').astype(float)
            x_parts.append(vals.to_frame(feat_name))
        else:
            cat = out_df[col].astype('object').where(out_df[col].notna(), 'Missing').astype(str)
            for feat_name in info.get('features') or []:
                level = str(feat_name).split('=', 1)[1] if '=' in str(feat_name) else str(feat_name)
                x_parts.append((cat == level).astype(float).to_frame(feat_name))
    X_df = pd.concat(x_parts, axis=1) if x_parts else pd.DataFrame(index=out_df.index)
    feature_cols = list(bundle.get('features') or bundle.get('feature_cols') or X_df.columns)
    for col in feature_cols:
        if col not in X_df.columns:
            X_df[col] = 0.0
    X_df = X_df[feature_cols].astype(float)
    coef = np.asarray(bundle.get('coef') or bundle.get('coefficients'), dtype=float)
    if bool(bundle.get('include_constant', True)):
        if len(coef) != len(feature_cols) + 1:
            raise ValueError('Regression Predictor: coefficient count does not match feature count')
        pred = X_df.to_numpy(dtype=float) @ coef[:-1] + coef[-1]
    else:
        if len(coef) != len(feature_cols):
            raise ValueError('Regression Predictor: coefficient count does not match feature count')
        pred = X_df.to_numpy(dtype=float) @ coef
else:
    est = bundle.get('estimator') or bundle.get('model')
    if est is None:
        raise ValueError('Regression Predictor: missing estimator/model bundle')
    feature_cols = list(bundle.get('features') or bundle.get('feature_cols') or getattr(est, 'feature_names_in_', []))
    if not feature_cols:
        feature_cols = [c for c in out_df.columns if c != target]
    missing = [c for c in feature_cols if c not in out_df.columns]
    if missing:
        raise KeyError(f'Regression Predictor: missing feature columns: {missing}')
    pred = est.predict(out_df[feature_cols])
out_df[pred_col] = pd.Series(pred, index=out_df.index).astype(float)
context['1574:1'] = out_df


## Column Filter \# `1571`
Node state: `EXECUTED`  
Input: [Port 1] 1574:1 from Regression Predictor #1574  
Output: [Port 1] 1571:1 to 1578 Math Formula #1578; [Port 1] 1571:1 to 1584 Color Manager #1584; [Port 1] 1571:1 to 1588 CSV Writer #1588  


In [61]:
################################################################################################################################################################
## Column Filter # `1571`
# Node state: `EXECUTED`
# Input: [Port 1] 1574:1 from Regression Predictor #1574
# Output: [Port 1] 1571:1 to 1578 Math Formula #1578; [Port 1] 1571:1 to 1584 Color Manager #1584; [Port 1] 1571:1 to 1588 CSV Writer #1588
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.preproc.filter.column2.ColumnFilter2NodeFactory
df = context['1574:1']  # input table
out_df = df
include_cols = ['Country', 'Approval rate', 'Applications_log10', 'Prediction (Approval rate)']
present = [c for c in include_cols if c in out_df.columns]
out_df = out_df.loc[:, present]
context['1571:1'] = out_df


## Numeric Scorer \# `1577`
Node state: `EXECUTED`  
Input: [Port 1] 1574:1 from Regression Predictor #1574  
Output: [Port 1] 1577:1 to 1590 CSV Writer #1590  


In [62]:
################################################################################################################################################################
## Numeric Scorer # `1577`
# Node state: `EXECUTED`
# Input: [Port 1] 1574:1 from Regression Predictor #1574
# Output: [Port 1] 1577:1 to 1590 CSV Writer #1590
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.mine.scorer.numeric2.NumericScorer2NodeFactory
df = context['1574:1']  # input table
_reference_col = 'Approval rate'
_predicted_col = 'Prediction (Approval rate)'
missing = [c for c in [_reference_col, _predicted_col] if c not in df.columns]
if missing:
    raise KeyError(f'Numeric Scorer: missing required column(s): {missing}')
pair = df[[_reference_col, _predicted_col]].copy()
pair[_reference_col] = pd.to_numeric(pair[_reference_col], errors='coerce')
pair[_predicted_col] = pd.to_numeric(pair[_predicted_col], errors='coerce')
pair = pair.dropna(subset=[_reference_col, _predicted_col])
y_true = pair[_reference_col].to_numpy(dtype=float)
y_pred = pair[_predicted_col].to_numpy(dtype=float)
err = y_true - y_pred
n = int(len(y_true))
if n == 0:
    r2 = mae = mse = rmse = mean_signed = mape = adjusted_r2 = float('nan')
else:
    sse = float(np.sum(err ** 2))
    centered = y_true - float(np.mean(y_true))
    sst = float(np.sum(centered ** 2))
    r2 = float(1.0 - sse / sst) if sst != 0 else float('nan')
    mae = float(np.mean(np.abs(err)))
    mse = float(np.mean(err ** 2))
    rmse = float(np.sqrt(mse))
    mean_signed = float(np.mean(err))
    if np.any(y_true == 0):
        mape = 'NaN'
    else:
        mape = float(np.mean(np.abs(err / y_true)))
    p = int(0)
    denom = n - p - 1
    adjusted_r2 = float(1.0 - (1.0 - r2) * (n - 1) / denom) if denom > 0 and pd.notna(r2) else float('nan')
_score_col = 'Prediction (Approval rate)'
out_df = pd.DataFrame({_score_col: [r2, mae, mse, rmse, mean_signed, mape, adjusted_r2]})
context['1577:1'] = out_df


## Math Formula \# `1578`
Node state: `EXECUTED`  
Input: [Port 1] 1571:1 from Column Filter #1571  
Output: [Port 1] 1578:1 to 1581 Row Splitter #1581  


In [63]:
################################################################################################################################################################
## Math Formula # `1578`
# Node state: `EXECUTED`
# Input: [Port 1] 1571:1 from Column Filter #1571
# Output: [Port 1] 1578:1 to 1581 Row Splitter #1581
# https://hub.knime.com/knime/extensions/org.knime.features.ext.jep/latest/org.knime.ext.jep.JEPNodeFactory
df = context['1571:1']  # input table
out_df = df.copy()
_expr = "df['Approval rate']-df['Prediction (Approval rate)']"  # translated from JEP
# Evaluate the expression in a restricted namespace
_ns = {'np': np, 'pd': pd, 'df': df}
_res = eval(_expr, {'__builtins__': {}}, _ns)
_target_col = 'Prediction_delta'
out_df[_target_col] = _res
context['1578:1'] = out_df


## Row Splitter \# `1581`
Node state: `EXECUTED`  
Input: [Port 1] 1578:1 from Math Formula #1578  
Output: [Port 1] 1581:1 to 1582 Math Formula #1582  


In [64]:
################################################################################################################################################################
## Row Splitter # `1581`
# Node state: `EXECUTED`
# Input: [Port 1] 1578:1 from Math Formula #1578
# Output: [Port 1] 1581:1 to 1582 Math Formula #1582
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.preproc.filter.row3.RowSplitterNodeFactory
df = context['1578:1']  # input table
def _rf_norm_name(s):
    return _re.sub(r'[^a-z0-9]+', '', str(s).lower())
_RF_LCMAP = {c.lower(): c for c in df.columns}
_RF_NORMMAP = {_rf_norm_name(c): c for c in df.columns}

def _rf_resolve(name):
    if name in df.columns:
        return name
    if name is None:
        return None
    c = _RF_LCMAP.get(str(name).lower())
    if c is not None:
        return c
    key = _rf_norm_name(name)
    c = _RF_NORMMAP.get(key)
    if c is not None:
        return c
    # Heuristic aliases for common aggregate column names
    if key in {'occurrencecount','count','rowcount'}:
        cand = [col for col in df.columns if 'count' in _rf_norm_name(col)]
        if len(cand) == 1:
            return cand[0]
    return None

def _rf_to_num(val):
    s = pd.Series([val])
    v = pd.to_numeric(s, errors='coerce').iloc[0]
    return v

mask = pd.Series(True, index=df.index)
_col0 = _rf_resolve('Approval rate')
if _col0 is None:
    _c0 = pd.Series(True, index=df.index)  # missing column → neutral
else:
    _s0 = df[_col0]
    _c0 = pd.to_numeric(_s0, errors='coerce') < _rf_to_num('0.9')
mask = (mask & _c0)
_invert = False
final_mask = (~mask) if _invert else mask
out_df = df[final_mask].copy()
_matching_df = out_df
_non_matching_df = df.loc[~final_mask].copy() if 'final_mask' in locals() else df.iloc[0:0].copy()
_port1_df = _matching_df
_port2_df = _non_matching_df
context['1581:1'] = _port1_df


## Math Formula \# `1582`
Node state: `EXECUTED`  
Input: [Port 1] 1581:1 from Row Splitter #1581  
Output: [Port 1] 1582:1 to 1583 Sorter #1583  


In [65]:
################################################################################################################################################################
## Math Formula # `1582`
# Node state: `EXECUTED`
# Input: [Port 1] 1581:1 from Row Splitter #1581
# Output: [Port 1] 1582:1 to 1583 Sorter #1583
# https://hub.knime.com/knime/extensions/org.knime.features.ext.jep/latest/org.knime.ext.jep.JEPNodeFactory
df = context['1581:1']  # input table
out_df = df.copy()
_expr = "df['Approval rate']/df['Prediction (Approval rate)']"  # translated from JEP
# Evaluate the expression in a restricted namespace
_ns = {'np': np, 'pd': pd, 'df': df}
_res = eval(_expr, {'__builtins__': {}}, _ns)
_target_col = 'Prediction_rate'
out_df[_target_col] = _res
context['1582:1'] = out_df


## Sorter \# `1583`
Node state: `EXECUTED`  
Input: [Port 1] 1582:1 from Math Formula #1582  
Output: [Port 1] 1583:1 to 1589 CSV Writer #1589  


In [66]:
################################################################################################################################################################
## Sorter # `1583`
# Node state: `EXECUTED`
# Input: [Port 1] 1582:1 from Math Formula #1582
# Output: [Port 1] 1583:1 to 1589 CSV Writer #1589
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.preproc.sorter.SorterNodeFactory
df = context['1582:1']  # input table
_sort_columns = ['Prediction_rate']
_sort_ascending = [True]
_missing_to_end = False
_existing_sort = [(col, asc) for col, asc in zip(_sort_columns, _sort_ascending) if col in df.columns]
if _existing_sort:
    if len(_existing_sort) == 1 and not _existing_sort[0][1] and pd.api.types.is_timedelta64_dtype(df[_existing_sort[0][0]]):
        _sort_col = _existing_sort[0][0]
        _minutes = df[_sort_col].dt.total_seconds() / 60
        _sign_rank = pd.Series(3, index=df.index)
        _sign_rank[_minutes < 0] = 0
        _sign_rank[_minutes > 0] = 1
        _sign_rank[_minutes == 0] = 2
        def _k2p_duration_sort_text(_value):
            if pd.isna(_value):
                return ''
            _total_minutes = int(pd.Timedelta(_value).total_seconds() // 60)
            if _total_minutes == 0:
                return 'PT0S'
            _sign = -1 if _total_minutes < 0 else 1
            _hours, _mins = divmod(abs(_total_minutes), 60)
            if _sign < 0:
                if _hours and _mins:
                    return f'PT-{_hours}H-{_mins}M'
                if _hours:
                    return f'PT-{_hours}H'
                return f'PT-{_mins}M'
            if _hours and _mins:
                return f'PT{_hours}H{_mins}M'
            if _hours:
                return f'PT{_hours}H'
            return f'PT{_mins}M'
        def _k2p_natural_key(_text):
            return re.sub(r'\d+', lambda _m: f'{int(_m.group()):012d}', str(_text))
        _duration_text = df[_sort_col].map(_k2p_duration_sort_text).map(_k2p_natural_key)
        out_df = (
            df.assign(_k2p_sign_rank=_sign_rank, _k2p_duration_text=_duration_text)
            .sort_values(['_k2p_sign_rank', '_k2p_duration_text'], ascending=[True, False], kind='mergesort')
            .drop(columns=['_k2p_sign_rank', '_k2p_duration_text'])
            .reset_index(drop=True)
        )
    else:
        out_df = df.sort_values(
            by=[col for col, _asc in _existing_sort],
            ascending=[asc for _col, asc in _existing_sort],
            na_position='last' if _missing_to_end else 'first',
            kind='mergesort',
        ).reset_index(drop=True)
else:
    out_df = df.copy()
context['1583:1'] = out_df


## Color Manager \# `1584`
Node state: `EXECUTED`  
Input: [Port 1] 1571:1 from Column Filter #1571  
Output: [Port 1] 1584:1 to 1572 Scatter Plot #1572  


In [67]:
################################################################################################################################################################
## Color Manager # `1584`
# Node state: `EXECUTED`
# Input: [Port 1] 1571:1 from Column Filter #1571
# Output: [Port 1] 1584:1 to 1572 Scatter Plot #1572
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.viz.property.color.ColorManager2NodeFactory
# Color metadata is not preserved in pandas; passthrough the table unchanged.
df = context['1571:1']
context['1584:1'] = df


## Scatter Plot \# `1572`
Node state: `EXECUTED`  
Input: [Port 1] 1584:1 from Color Manager #1584  


In [68]:
################################################################################################################################################################
## Scatter Plot # `1572`
# Node state: `EXECUTED`
# Input: [Port 1] 1584:1 from Color Manager #1584
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.views.node.scatterplot.ScatterPlotNodeFactory
df = context['1584:1']  # input table
_x_col = 'Prediction (Approval rate)'
_y_col = 'Approval rate'
_color_col = 'Applications_log10'
_title = 'Scatter Plot'
_x_label = 'Prediction (Approval rate)'
_y_label = 'Approval rate'
_width_in = 800 / 100.0
_height_in = 600 / 100.0
_img_fmt = 'SVG'
_max_rows = int(2500)
_point_size = int(15)
_show_legend = True
_axis_extent_method = 'AUTO'
_manual_xlim = (0.0, 100.0)
_manual_ylim = (0.0, 100.0)

if _x_col is None or _x_col not in df.columns:
    raise KeyError(f'Scatter Plot: x-axis column not found: {_x_col!r}')
if _y_col is None or _y_col not in df.columns:
    raise KeyError(f'Scatter Plot: y-axis column not found: {_y_col!r}')
plot_df = df.head(_max_rows).copy() if _max_rows > 0 else df.copy()
x = pd.to_numeric(plot_df[_x_col], errors='coerce')
y = pd.to_numeric(plot_df[_y_col], errors='coerce')
valid = x.notna() & y.notna()
plot_df = plot_df.loc[valid].copy()
x = x.loc[valid]
y = y.loc[valid]

_green_blue = LinearSegmentedColormap.from_list('knime2py_green_blue', ['#2ca25f', '#2b8cbe'])
color_values = None
if _color_col and _color_col in plot_df.columns:
    c_num = pd.to_numeric(plot_df[_color_col], errors='coerce')
    if c_num.notna().any():
        color_values = c_num
    else:
        color_values = pd.Series(pd.factorize(plot_df[_color_col].astype('string'))[0], index=plot_df.index)

fig, ax = plt.subplots(figsize=(_width_in, _height_in), dpi=100)
if color_values is not None:
    sc = ax.scatter(x, y, c=color_values, cmap=_green_blue, s=_point_size, alpha=0.85, edgecolors='none')
    if _show_legend:
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label(_color_col)
else:
    ax.scatter(x, y, color='#2b8cbe', s=_point_size, alpha=0.85, edgecolors='none')
ax.set_title(_title)
ax.set_xlabel(_x_label)
ax.set_ylabel(_y_label)
if _axis_extent_method == 'MANUAL':
    ax.set_xlim(*_manual_xlim)
    ax.set_ylim(*_manual_ylim)
ax.grid(True, linewidth=0.5, alpha=0.35)
fig.tight_layout()

try:
    out_dir = Path.cwd()
    out_dir.mkdir(parents=True, exist_ok=True)
except Exception:
    out_dir = Path(tempfile.gettempdir()) / 'knime2py_scatter'
    out_dir.mkdir(parents=True, exist_ok=True)
ext = 'svg' if str(_img_fmt).upper() == 'SVG' else 'png'
img_path = out_dir / ('data/results/approximation_sp.' + ext)
fig.savefig(img_path, bbox_inches='tight')
plt.close(fig)
print(f'[Scatter Plot] Wrote image to: {img_path}')


[Scatter Plot] Wrote image to: /Users/vitaly/Home/git_proj/2026-04-INZ-Visa-decisions/data/results/approximation_sp.svg


## CSV Writer \# `1587`
Node state: `CONFIGURED`  
Input: [Port 1] 1576:2 from Linear Regression Learner #1576  


In [69]:
################################################################################################################################################################
## CSV Writer # `1587`
# Node state: `CONFIGURED`
# Input: [Port 1] 1576:2 from Linear Regression Learner #1576
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.io.filehandling.csv.writer.CSVWriter2NodeFactory
df = context['1576:2']
out_path = Path(r"data/results/Coefficients.csv")
df = df.copy()
for _col in df.select_dtypes(include=['datetime', 'datetimetz']).columns:
    df[_col] = df[_col].dt.strftime('%Y-%m-%dT%H:%M')
def _k2p_format_timedelta(_value):
    if pd.isna(_value):
        return pd.NA
    _total_minutes = int(pd.Timedelta(_value).total_seconds() // 60)
    if _total_minutes == 0:
        return 'PT0S'
    _sign = -1 if _total_minutes < 0 else 1
    _hours, _minutes = divmod(abs(_total_minutes), 60)
    if _sign < 0:
        if _hours and _minutes:
            return f'PT-{_hours}H-{_minutes}M'
        if _hours:
            return f'PT-{_hours}H'
        return f'PT-{_minutes}M'
    if _hours and _minutes:
        return f'PT{_hours}H{_minutes}M'
    if _hours:
        return f'PT{_hours}H'
    return f'PT{_minutes}M'
for _col in df.select_dtypes(include=['timedelta']).columns:
    df[_col] = df[_col].map(_k2p_format_timedelta)
for _col in df.select_dtypes(include=['float']).columns:
    _series = df[_col].dropna()
    if not _series.empty and ((_series % 1) == 0).all():
        df[_col] = df[_col].astype('Int64')
_k2p_missing_sentinel = '__K2P_MISSING_FIELD_9b1f6f7c__'
df = df.astype('object').mask(df.isna(), _k2p_missing_sentinel)
df.to_csv(out_path, sep=',', quotechar='"', header=True, encoding='UTF-8', na_rep='', index=False, quoting=csv.QUOTE_NONNUMERIC)
_k2p_csv_text = out_path.read_text(encoding='UTF-8')
_k2p_csv_text = _k2p_csv_text.replace('"' + _k2p_missing_sentinel + '"', '')
out_path.write_text(_k2p_csv_text, encoding='UTF-8')


2748

## CSV Writer \# `1588`
Node state: `CONFIGURED`  
Input: [Port 1] 1571:1 from Column Filter #1571  


In [70]:
################################################################################################################################################################
## CSV Writer # `1588`
# Node state: `CONFIGURED`
# Input: [Port 1] 1571:1 from Column Filter #1571
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.io.filehandling.csv.writer.CSVWriter2NodeFactory
df = context['1571:1']
out_path = Path(r"data/results/Prediction.csv")
df = df.copy()
for _col in df.select_dtypes(include=['datetime', 'datetimetz']).columns:
    df[_col] = df[_col].dt.strftime('%Y-%m-%dT%H:%M')
def _k2p_format_timedelta(_value):
    if pd.isna(_value):
        return pd.NA
    _total_minutes = int(pd.Timedelta(_value).total_seconds() // 60)
    if _total_minutes == 0:
        return 'PT0S'
    _sign = -1 if _total_minutes < 0 else 1
    _hours, _minutes = divmod(abs(_total_minutes), 60)
    if _sign < 0:
        if _hours and _minutes:
            return f'PT-{_hours}H-{_minutes}M'
        if _hours:
            return f'PT-{_hours}H'
        return f'PT-{_minutes}M'
    if _hours and _minutes:
        return f'PT{_hours}H{_minutes}M'
    if _hours:
        return f'PT{_hours}H'
    return f'PT{_minutes}M'
for _col in df.select_dtypes(include=['timedelta']).columns:
    df[_col] = df[_col].map(_k2p_format_timedelta)
for _col in df.select_dtypes(include=['float']).columns:
    _series = df[_col].dropna()
    if not _series.empty and ((_series % 1) == 0).all():
        df[_col] = df[_col].astype('Int64')
_k2p_missing_sentinel = '__K2P_MISSING_FIELD_9b1f6f7c__'
df = df.astype('object').mask(df.isna(), _k2p_missing_sentinel)
df.to_csv(out_path, sep=',', quotechar='"', header=True, encoding='UTF-8', na_rep='', index=False, quoting=csv.QUOTE_NONNUMERIC)
_k2p_csv_text = out_path.read_text(encoding='UTF-8')
_k2p_csv_text = _k2p_csv_text.replace('"' + _k2p_missing_sentinel + '"', '')
out_path.write_text(_k2p_csv_text, encoding='UTF-8')


7226

## CSV Writer \# `1589`
Node state: `CONFIGURED`  
Input: [Port 1] 1583:1 from Sorter #1583  


In [71]:
################################################################################################################################################################
## CSV Writer # `1589`
# Node state: `CONFIGURED`
# Input: [Port 1] 1583:1 from Sorter #1583
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.io.filehandling.csv.writer.CSVWriter2NodeFactory
df = context['1583:1']
out_path = Path(r"data/results/Prediction_rate.csv")
df = df.copy()
for _col in df.select_dtypes(include=['datetime', 'datetimetz']).columns:
    df[_col] = df[_col].dt.strftime('%Y-%m-%dT%H:%M')
def _k2p_format_timedelta(_value):
    if pd.isna(_value):
        return pd.NA
    _total_minutes = int(pd.Timedelta(_value).total_seconds() // 60)
    if _total_minutes == 0:
        return 'PT0S'
    _sign = -1 if _total_minutes < 0 else 1
    _hours, _minutes = divmod(abs(_total_minutes), 60)
    if _sign < 0:
        if _hours and _minutes:
            return f'PT-{_hours}H-{_minutes}M'
        if _hours:
            return f'PT-{_hours}H'
        return f'PT-{_minutes}M'
    if _hours and _minutes:
        return f'PT{_hours}H{_minutes}M'
    if _hours:
        return f'PT{_hours}H'
    return f'PT{_minutes}M'
for _col in df.select_dtypes(include=['timedelta']).columns:
    df[_col] = df[_col].map(_k2p_format_timedelta)
for _col in df.select_dtypes(include=['float']).columns:
    _series = df[_col].dropna()
    if not _series.empty and ((_series % 1) == 0).all():
        df[_col] = df[_col].astype('Int64')
_k2p_missing_sentinel = '__K2P_MISSING_FIELD_9b1f6f7c__'
df = df.astype('object').mask(df.isna(), _k2p_missing_sentinel)
df.to_csv(out_path, sep=',', quotechar='"', header=True, encoding='UTF-8', na_rep='', index=False, quoting=csv.QUOTE_NONNUMERIC)
_k2p_csv_text = out_path.read_text(encoding='UTF-8')
_k2p_csv_text = _k2p_csv_text.replace('"' + _k2p_missing_sentinel + '"', '')
out_path.write_text(_k2p_csv_text, encoding='UTF-8')


6881

## CSV Writer \# `1590`
Node state: `CONFIGURED`  
Input: [Port 1] 1577:1 from Numeric Scorer #1577  


In [72]:
################################################################################################################################################################
## CSV Writer # `1590`
# Node state: `CONFIGURED`
# Input: [Port 1] 1577:1 from Numeric Scorer #1577
# https://hub.knime.com/knime/extensions/org.knime.features.base/latest/org.knime.base.node.io.filehandling.csv.writer.CSVWriter2NodeFactory
df = context['1577:1']
out_path = Path(r"data/results/Score.csv")
df = df.copy()
for _col in df.select_dtypes(include=['datetime', 'datetimetz']).columns:
    df[_col] = df[_col].dt.strftime('%Y-%m-%dT%H:%M')
def _k2p_format_timedelta(_value):
    if pd.isna(_value):
        return pd.NA
    _total_minutes = int(pd.Timedelta(_value).total_seconds() // 60)
    if _total_minutes == 0:
        return 'PT0S'
    _sign = -1 if _total_minutes < 0 else 1
    _hours, _minutes = divmod(abs(_total_minutes), 60)
    if _sign < 0:
        if _hours and _minutes:
            return f'PT-{_hours}H-{_minutes}M'
        if _hours:
            return f'PT-{_hours}H'
        return f'PT-{_minutes}M'
    if _hours and _minutes:
        return f'PT{_hours}H{_minutes}M'
    if _hours:
        return f'PT{_hours}H'
    return f'PT{_minutes}M'
for _col in df.select_dtypes(include=['timedelta']).columns:
    df[_col] = df[_col].map(_k2p_format_timedelta)
for _col in df.select_dtypes(include=['float']).columns:
    _series = df[_col].dropna()
    if not _series.empty and ((_series % 1) == 0).all():
        df[_col] = df[_col].astype('Int64')
_k2p_missing_sentinel = '__K2P_MISSING_FIELD_9b1f6f7c__'
df = df.astype('object').mask(df.isna(), _k2p_missing_sentinel)
df.to_csv(out_path, sep=',', quotechar='"', header=True, encoding='UTF-8', na_rep='', index=False, quoting=csv.QUOTE_NONNUMERIC)
_k2p_csv_text = out_path.read_text(encoding='UTF-8')
_k2p_csv_text = _k2p_csv_text.replace('"' + _k2p_missing_sentinel + '"', '')
out_path.write_text(_k2p_csv_text, encoding='UTF-8')


156